# 02 — Serialization QC before embeddings

This notebook validates longitudinal construction and temporal falsifications before embedding computation. It does not download or run an embedding model.

## Rules and condition matrix

Calendar days are atomic clinical blocks because this MEDS conversion has no reliable encounter identifier. Same-time events are concurrent and receive only a deterministic tie-break. The four full-history conditions preserve clinical content and order. `latest_window_only` is a separate information-reduction baseline, not a row of the full-history condition matrix.

In [ ]:
import datetime as dt
from pathlib import Path

from IPython.display import display

from temporal_ehr.config import resolve_path
from temporal_ehr.data.task_registry import load_task_registry
from temporal_ehr.serialization.blocks import ClinicalEvent, build_calendar_blocks
from temporal_ehr.serialization.conditions import FULL_HISTORY_CONDITIONS, build_full_history_conditions
from temporal_ehr.serialization.packing import load_qwen_tokenizer, whitespace_estimate_for_synthetic_demo
from temporal_ehr.serialization.qc import run_serialization_qc
from temporal_ehr.serialization.render import render_condition

display(
    {
        "full_history_conditions": FULL_HISTORY_CONDITIONS,
        "latest_window_only_is_separate": True,
        "atomic_block": "calendar day",
        "qc_aggregation_windows_days": [30, 90],
        "primary_window_selected": False,
    }
)

## Entirely synthetic timeline demonstration

Only the invented timeline below is rendered. Its whitespace estimate is not a tokenizer token count.

In [ ]:
synthetic_prediction_time = dt.datetime(2030, 1, 1, 12)
synthetic_events = (
    ClinicalEvent(dt.datetime(2028, 1, 1, 9), "SYNTHETIC/ALPHA", numeric_value=1.0, unit="invented-unit"),
    ClinicalEvent(dt.datetime(2029, 1, 1, 9), "SYNTHETIC/RECURRENT"),
    ClinicalEvent(dt.datetime(2029, 1, 1, 9), "SYNTHETIC/CONCURRENT"),
    ClinicalEvent(dt.datetime(2029, 9, 1, 10), "SYNTHETIC/RECURRENT"),
    ClinicalEvent(dt.datetime(2029, 12, 20, 8), "SYNTHETIC/OMEGA"),
)
synthetic_blocks = build_calendar_blocks(synthetic_events, synthetic_prediction_time)
synthetic_conditions = build_full_history_conditions(synthetic_blocks, seed=20260815, instance_key="entirely-synthetic")
synthetic_descriptions = {
    "SYNTHETIC/ALPHA": "Invented alpha event",
    "SYNTHETIC/RECURRENT": "Invented recurrent event",
    "SYNTHETIC/CONCURRENT": "Invented concurrent event",
    "SYNTHETIC/OMEGA": "Invented omega event",
}
synthetic_display = {}
for name, condition in synthetic_conditions.items():
    text = render_condition(condition, synthetic_descriptions)
    synthetic_display[name] = {
        "rendered_entirely_synthetic_history": text,
        "whitespace_estimate_not_token_count": whitespace_estimate_for_synthetic_demo(text),
    }
display(synthetic_display)

## Real-data aggregate QC configuration

Real histories are streamed, summarized, and discarded without display. Exact packing requires an already-downloaded local Qwen tokenizer; the notebook fails clearly if it is unavailable.

In [ ]:
event_path = resolve_path(None, "EHRSHOT_DATA_PATH", must_be_file=True)
asset_root = resolve_path(None, "EHRSHOT_ASSET_ROOT", must_be_dir=True)
qwen_path = resolve_path(None, "QWEN_EMBEDDING_PATH", must_be_dir=True)
cohort_path = Path("outputs/private/cohorts/primary_panel.parquet")
if not cohort_path.is_file():
    command = (
        ".venv/bin/python -m temporal_ehr.cli build-cohort "
        '--asset-root "${EHRSHOT_ASSET_ROOT}" --tasks lab_anemia new_acutemi guo_los '
        "--selection-seed 20260814 --private-output outputs/private/cohorts/primary_panel.parquet "
        "--allow-private-output"
    )
    print(f"Missing governed cohort. Rebuild it exactly with:\n{command}")
    raise FileNotFoundError("primary_panel.parquet is required; no replacement cohort was created")
tasks = tuple(load_task_registry(Path("configs/tasks.yaml"))["primary_panel"])
tokenizer = load_qwen_tokenizer(qwen_path)
display(
    {
        "tasks": tasks,
        "splits": ["train", "validation"],
        "token_budget": 4096,
        "perturbation_seed": 20260815,
        "logical_tokenizer_directory": qwen_path.name,
    }
)

## Aggregate real-data distributions

In [ ]:
qc_report = run_serialization_qc(
    event_path,
    cohort_path,
    asset_root,
    tokenizer=tokenizer,
    tasks=tasks,
    splits=("train", "validation"),
    token_budget=4096,
    perturbation_seed=20260815,
)
display(qc_report)

## QC interpretation

The 30-day and 90-day block aggregations are QC comparisons only; no final primary window is selected here. `latest_window_only` deliberately discards earlier clinical content and remains separate from the four complete-trajectory conditions. All reported real-data groups apply minimum size 20.